# Cell 1: Install Required Libraries

In [1]:
!pip install datasets pandas transformers torch

# Cell 2: Import Dependencies

In [2]:
import pandas as pd
from datasets import load_dataset

# Set pandas display options to easily read long text strings in the notebook
pd.set_option('display.max_colwidth', None)

# Cell 3: Import Hausa dataset

In [3]:
# Load the BRIGHTER dataset specifically for the Hausa configuration ("hau")
hausa_dataset = load_dataset("brighter-dataset/BRIGHTER-emotion-categories", "hau")

# View the structure of the downloaded Hausa dataset
print("Hausa Dataset Splits:")
print(hausa_dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

hau/train-00000-of-00001.parquet:   0%|          | 0.00/144k [00:00<?, ?B/s]

hau/dev-00000-of-00001.parquet:   0%|          | 0.00/31.3k [00:00<?, ?B/s]

hau/test-00000-of-00001.parquet:   0%|          | 0.00/83.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2145 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/712 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2160 [00:00<?, ? examples/s]

Hausa Dataset Splits:
DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'emotions'],
        num_rows: 2145
    })
    dev: Dataset({
        features: ['id', 'text', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'emotions'],
        num_rows: 712
    })
    test: Dataset({
        features: ['id', 'text', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'emotions'],
        num_rows: 2160
    })
})


# Cell 4: Inspect data  
Since hausa_dataset is already filtered, you can jump straight to inspecting it:

In [4]:
# Convert the Hausa training split to a Pandas DataFrame for visual inspection
df_hausa_train = hausa_dataset['train'].to_pandas()

# Display the first 5 rows
df_hausa_train.head()

,id,text,anger,disgust,fear,joy,sadness,surprise,emotions
0,hau_train_track_a_00001,"kotu ta yi hukunci kan shari'ar zaben dan majalisar pdp, ta yi hukuncin bazata",0,0,0,0,0,1,[surprise]
1,hau_train_track_a_00002,"toh fah inji 'yan magana suka ce """"""""ana wata ga wata""""""""🤣😂",0,0,0,0,0,1,[surprise]
2,hau_train_track_a_00003,bincike ya nuna yan najeriya sun fi damuwa da rashin tsaro da talauci fiye da korona,0,0,1,0,1,0,"[fear, sadness]"
3,hau_train_track_a_00004,kwamishina ya musanta rahoton masari ya cire kusan miliyan n500m don tarban buhari,0,0,0,0,0,0,[]
4,hau_train_track_a_00005,innalillahi wa inna ilaihir raji'un: allah ya yi wa mahaifiyar halima atete rasuwa,0,0,0,0,1,0,[sadness]


# Cell 5: Basic Text Cleaning  
Even though the BRIGHTER dataset is generally high-quality, it is always a good idea to ensure there are no lingering URLs, excessive spaces, or weird formatting artifacts before passing the text to a transformer model.

In [5]:
import re

def clean_text(example):
    text = example['text']
    # Remove URLs if any exist
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove extra whitespaces
    text = text.strip()
    return {'text': text}

# Apply the cleaning function to all splits
hausa_dataset = hausa_dataset.map(clean_text)

print("Text cleaning complete!")

Map:   0%|          | 0/2145 [00:00<?, ? examples/s]

Map:   0%|          | 0/712 [00:00<?, ? examples/s]

Map:   0%|          | 0/2160 [00:00<?, ? examples/s]

Text cleaning complete!


# Cell 6: Formatting the Multilabel Target Vectors  
To train a multilabel model, the individual binary columns (anger, disgust, fear, joy, sadness, surprise) must be combined into a single array (vector) for each row. Furthermore, PyTorch's Binary Cross-Entropy loss requires these labels to be formatted as floats, not integers.

In [6]:
# Define the exact order of your emotion columns
emotion_cols = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']

def format_labels(example):
    # Create a list of floats representing the presence (1.0) or absence (0.0) of each emotion
    labels = [float(example[col]) for col in emotion_cols]
    return {'labels': labels}

# Apply the label formatting to all splits
hausa_dataset = hausa_dataset.map(format_labels)

# Inspect the first row of the training set to verify the new 'labels' column
print("Original emotions list:", hausa_dataset['train'][0]['emotions'])
print("New target vector (labels):", hausa_dataset['train'][0]['labels'])

Map:   0%|          | 0/2145 [00:00<?, ? examples/s]

Map:   0%|          | 0/712 [00:00<?, ? examples/s]

Map:   0%|          | 0/2160 [00:00<?, ? examples/s]

Original emotions list: ['surprise']
New target vector (labels): [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]


# Cell 7: Tokenization (Preparing for AfroXLMR / XLM-RoBERTa)  
Now that the text is clean and the labels are perfectly formatted, we need to convert the text into tokens (numbers) that the model can actually understand. We are using AfroXLMR. We will pull its specific tokenizer here.

In [7]:
from transformers import AutoTokenizer

# Load the tokenizer for AfroXLMR (or swap with "xlm-roberta-large" if preferred)
model_checkpoint = "Davlan/afro-xlmr-large"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    # Tokenize the text, pad shorter sentences, and truncate longer ones to a max length
    return tokenizer(
        examples['text'],
        padding="max_length",
        truncation=True,
        max_length=128 # 128 is usually sufficient for tweets/short texts; increase to 256 or 512 if your texts are long paragraphs
    )

# Apply tokenization in batches (this is much faster)
tokenized_hausa = hausa_dataset.map(tokenize_function, batched=True)

# View the new columns added by the tokenizer
print("Columns after tokenization:", tokenized_hausa['train'].column_names)

config.json:   0%|          | 0.00/714 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/399 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Map:   0%|          | 0/2145 [00:00<?, ? examples/s]

Map:   0%|          | 0/712 [00:00<?, ? examples/s]

Map:   0%|          | 0/2160 [00:00<?, ? examples/s]

Columns after tokenization: ['id', 'text', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'emotions', 'labels', 'input_ids', 'attention_mask']


# Cell 8: Final Cleanup for PyTorch  
The Hugging Face Trainer expects only specific columns (input_ids, attention_mask, and labels). It will throw an error if we leave the raw string text or the old integer columns in the dataset during training.

In [8]:
# Define the columns we want to KEEP for training
columns_to_keep = ['input_ids', 'attention_mask', 'labels']

# Remove all other columns (like 'id', 'text', 'anger', 'joy', etc.)
tokenized_hausa.set_format(type='torch', columns=columns_to_keep)

print("Dataset is fully pre-processed and ready for PyTorch!")
print(tokenized_hausa['train'][0])

Dataset is fully pre-processed and ready for PyTorch!
{'labels': tensor([0., 0., 0., 0., 0., 1.]), 'input_ids': tensor([     0,   1975,     34,    308,  11180,  39545,   7700,    203, 152093,
            25,    147,     80,    776,    123, 185841,    915,     71,    254,
             4,    308,  11180,  39545,  78479,  12170,    102,      2,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
        

In [9]:
!pip install scikit-learn

In [10]:
# Check a sample
sample = tokenized_hausa['train'][0]
print(f"Labels type: {type(sample['labels'][0])}") # Should say <class 'torch.Tensor'>
print(f"Labels values: {sample['labels']}")       # Should look like tensor([0., 1., 0., ...])

Labels type: <class 'torch.Tensor'>
Labels values: tensor([0., 0., 0., 0., 0., 1.])


# Cell 9: Define the Evaluation Metrics  
Because this is a multilabel task, the model won't output a single probability. It will output "logits" (raw scores) for all 6 emotions. We need to pass these through a Sigmoid function and set a threshold (usually 0.5) to decide if an emotion is present (1) or not (0).

In [11]:
import numpy as np
import torch
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Apply sigmoid activation to the raw logits to get probabilities between 0 and 1
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(logits))

    # Convert probabilities to binary predictions using a 0.5 threshold
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= 0.3)] = 1

    # Calculate your core metrics
    f1_macro = f1_score(labels, y_pred, average='macro') # Your main project metric!
    f1_micro = f1_score(labels, y_pred, average='micro')
    accuracy = accuracy_score(labels, y_pred)

    return {
        'f1_macro': f1_macro,
        'f1_micro': f1_micro,
        'accuracy': accuracy
    }
print("Metrics defined!")

Metrics defined!


# Cell 10: Initialize the AfroXLMR Model  
We will now load the pre-trained transformer and add a classification head on top of it designed specifically for 6 labels. Setting problem_type="multi_label_classification" automatically configures the model to use the Binary Cross-Entropy (BCE) loss function we specified in our methodology.

In [12]:
from transformers import AutoModelForSequenceClassification

# We have 6 specific emotion categories
num_labels = 6
model_checkpoint = "Davlan/afro-xlmr-base"

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True # Best practice when initializing a new classification head
)
print("AfroXLMR model loaded and configured for multilabel classification!")

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


AfroXLMR model loaded and configured for multilabel classification!


# Cell 11: Set up the Training Arguments and Trainer  
The Trainer handles the heavy lifting of the PyTorch training loop. We will configure it to train for a few epochs and evaluate itself using the validation set after every epoch.

In [13]:
from transformers import TrainingArguments, Trainer

# Check the exact name of your validation split (sometimes it's 'dev' or 'validation')
val_split_name = 'validation' if 'validation' in tokenized_hausa else 'dev'

training_args = TrainingArguments(
    output_dir='./hausa_baseline_results',
    eval_strategy="epoch",          # Evaluate every epoch
    save_strategy="epoch",          # ADD THIS: Save every epoch so they match
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_hausa['train'],
    eval_dataset=tokenized_hausa[val_split_name],
    compute_metrics=compute_metrics
)
print("Trainer successfully initialized!")

Trainer successfully initialized!


# Cell 12: Train and Evaluate the Baseline!  
Running this cell will start the training process. Warning: Depending on your hardware (CPU vs. GPU), this step can take some time.

In [14]:
# 1. Start the training loop
print("Starting training...")
trainer.train()

# 2. Evaluate the model on the unseen TEST set to get your final Baseline metrics
print("\nEvaluating on Test Set...")
baseline_results = trainer.evaluate(tokenized_hausa['test'])

print("\n=== FINAL HAUSA BASELINE RESULTS ===")
print(f"F1-Macro: {baseline_results['eval_f1_macro']:.4f}")
print(f"F1-Micro: {baseline_results['eval_f1_micro']:.4f}")
print(f"Accuracy: {baseline_results['eval_accuracy']:.4f}")

Starting training...


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Accuracy
1,No log,0.350692,0.526618,0.577675,0.323034
2,No log,0.293436,0.665530,0.664352,0.432584
3,No log,0.274575,0.672697,0.675914,0.474719
4,0.326995,0.251198,0.733200,0.732769,0.550562
5,0.326995,0.253278,0.714446,0.714634,0.530899
6,0.326995,0.270921,0.709454,0.707547,0.494382
7,0.326995,0.266306,0.730319,0.726610,0.519663
8,0.154181,0.266566,0.738050,0.730816,0.522472
9,0.154181,0.270982,0.738253,0.734940,0.539326
10,0.154181,0.271478,0.728711,0.726404,0.519663


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Evaluating on Test Set...



=== FINAL HAUSA BASELINE RESULTS ===
F1-Macro: 0.6732
F1-Micro: 0.6763
Accuracy: 0.4731


In [15]:
# Check how many times each emotion appears in the training set
for i, col in enumerate(emotion_cols):
    count = sum([1 for x in tokenized_hausa['train']['labels'] if x[i] == 1.0])
    print(f"{col}: {count} positive examples")

anger: 408 positive examples
disgust: 329 positive examples
fear: 327 positive examples
joy: 320 positive examples
sadness: 647 positive examples
surprise: 349 positive examples


# Cell 13: Install Translation Dependencies  
We will need the sentencepiece library to handle NLLB’s tokenization.

In [16]:
!pip install sentencepiece

In [17]:
import torch
import gc

# Delete the old model and trainer to free up space
if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer

# Clear the cache
gc.collect()
torch.cuda.empty_cache()

print("GPU Memory Cleared!")

GPU Memory Cleared!


# Cell 14: Set up the NLLB Translation Pipeline  
We will use the "distilled" 600M version of NLLB. It is fast and fits easily in Google Colab's memory while providing high-quality translations.

In [18]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "facebook/nllb-200-distilled-600M"

# Load model in float16 to save memory
model_nllb = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).to("cuda")

tokenizer_nllb = AutoTokenizer.from_pretrained(model_name)

def translate_nllb(text, tgt_lang_code):
    inputs = tokenizer_nllb(text, return_tensors="pt").to("cuda")

    tgt_lang_id = tokenizer_nllb.convert_tokens_to_ids(tgt_lang_code)

    translated_tokens = model_nllb.generate(
        **inputs,
        forced_bos_token_id=tgt_lang_id,
        max_length=400
    )

    return tokenizer_nllb.batch_decode(translated_tokens, skip_special_tokens=True)[0]

def back_translate(text):
    if not text or text.strip() == "":
        return text
    try:
        tokenizer_nllb.src_lang = "hau_Latn"
        pivot_text = translate_nllb(text, "eng_Latn")

        tokenizer_nllb.src_lang = "eng_Latn"
        augmented_text = translate_nllb(pivot_text, "hau_Latn")

        return augmented_text
    except Exception as e:
        return f"RETRY_ERROR: {str(e)}"

# TEST IT
test_text = "kotu ta yi hukunci kan shari'ar zaben dan majalisar pdp"
print(f"Test Result: {back_translate(test_text)}")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Test Result: Kotuna ne ke yanke hukunci kan dokar zabe da kuma majalisar PDP.


# Cell 15: Generate Augmented Data  
We will need to expand your dataset to 1.5x and 2x.

To get to 2x, we will back-translate every row in your training set.

We will only augment the Training Set. Never augment the Test set, as that would "leak" information and ruin your evaluation.

In [19]:
import pandas as pd
from tqdm import tqdm
tqdm.pandas() # For a progress bar

# Convert training set back to dataframe if it isn't already
df_hausa_train = hausa_dataset['train'].to_pandas()

print("Starting back-translation (this will take a few minutes)...")
# Apply back-translation to the text column
df_hausa_train['augmented_text'] = df_hausa_train['text'].progress_apply(back_translate)

# Create the augmented dataframe
# We keep the same labels but use the new augmented_text
df_augmented = df_hausa_train.copy()
df_augmented['text'] = df_augmented['augmented_text']
df_augmented = df_augmented.drop(columns=['augmented_text'])

# Combine original + augmented for the 2x dataset
df_hausa_2x = pd.concat([df_hausa_train.drop(columns=['augmented_text']), df_augmented], ignore_index=True)

print(f"Original size: {len(df_hausa_train)}")
print(f"Augmented size (Condition B - 2x): {len(df_hausa_2x)}")

Starting back-translation (this will take a few minutes)...


100%|██████████| 2145/2145 [31:09<00:00,  1.15it/s]

Original size: 2145
Augmented size (Condition B - 2x): 4290


# Cell 16: Prepare the Augmented Dataset for Training  
Now we convert this new dataframe back into a format the Hugging Face Trainer can use.

In [20]:
from datasets import Dataset, DatasetDict

# Convert the pandas dataframe back to a Hugging Face Dataset
hausa_2x_dataset = Dataset.from_pandas(df_hausa_2x)

# Re-tokenize the 2x dataset (using the same function from Cell 7)
tokenized_hausa_2x = hausa_2x_dataset.map(tokenize_function, batched=True)

# Format for PyTorch
tokenized_hausa_2x.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Create a new DatasetDict using 'dev' instead of 'validation'
augmented_bundle = DatasetDict({
    'train': tokenized_hausa_2x,
    'dev': tokenized_hausa['dev'], # Changed from 'validation'
    'test': tokenized_hausa['test']
})

Map:   0%|          | 0/4290 [00:00<?, ? examples/s]

# Cell 17: Visual Check of Augmented Data  
Before training, it is crucial to verify that the back-translation actually changed the text while preserving the meaning.

In [21]:
from datasets import Dataset, DatasetDict

# 1. Re-convert the pandas dataframe (which should still be in memory)
hausa_2x_dataset = Dataset.from_pandas(df_hausa_2x)

# 2. Re-tokenize (Ensure tokenize_function is defined from Cell 7)
tokenized_hausa_2x = hausa_2x_dataset.map(tokenize_function, batched=True)

# 3. Format for PyTorch
tokenized_hausa_2x.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# 4. Re-create the bundle
augmented_bundle = DatasetDict({
    'train': tokenized_hausa_2x,
    'dev': tokenized_hausa['dev'],
    'test': tokenized_hausa['test']
})

print("Augmented bundle recreated! Ready to train.")

Map:   0%|          | 0/4290 [00:00<?, ? examples/s]

Augmented bundle recreated! Ready to train.


In [22]:
print("--- ORIGINAL HAUSA ---")
print(df_hausa_train['text'].iloc[0])

print("\n--- BACK-TRANSLATED HAUSA (Condition B) ---")
print(df_augmented['text'].iloc[0])

# Check if the labels remained consistent
print("\n--- LABELS (Should be identical) ---")
print(f"Original: {hausa_dataset['train'][0]['labels']}")
print(f"Augmented: {hausa_2x_dataset[len(df_hausa_train)]['labels']}")

--- ORIGINAL HAUSA ---
kotu ta yi hukunci kan shari'ar zaben dan majalisar pdp, ta yi hukuncin bazata

--- BACK-TRANSLATED HAUSA (Condition B) ---
Kotun ta yanke hukunci a kan batun zaben da kuma majalisar PDP, kuma ta yanke hukunci bisa ga hukuncin.

--- LABELS (Should be identical) ---
Original: [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
Augmented: [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]


# Cell 18: Re-initialize Model and Trainer for Condition B  
We must re-initialize the model to "reset" its brain. If we don't, we would be fine-tuning a model that already knows the test set, which is "cheating" (weight leakage).

In [23]:
#!rm -rf ./hausa_baseline_results
#!rm -rf ./hausa_aug_condition_b_results

In [24]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Reset the model to its pre-trained state (AfroXLMR-Base)
model_aug = AutoModelForSequenceClassification.from_pretrained(
    "Davlan/afro-xlmr-base",
    num_labels=6,
    problem_type="multi_label_classification"
)

# Use the same arguments as the baseline for a fair experiment
training_args_aug = TrainingArguments(
    output_dir='./hausa_aug_condition_b_results',
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,             # ADD THIS: Only keep the 2 most recent/best models
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro"
)

# Initialize the Trainer with the AUGMENTED training set
trainer_aug = Trainer(
    model=model_aug,
    args=training_args_aug,
    train_dataset=augmented_bundle['train'], # Using the 2x data here!
    eval_dataset=augmented_bundle['dev'],
    compute_metrics=compute_metrics # Using the function from Cell 9
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [25]:
import torch
import gc

# Delete the translation model and tokenizer to free up space for training
if 'model_nllb' in globals():
    del model_nllb
if 'tokenizer_nllb' in globals():
    del tokenizer_nllb

# Clear the cache
gc.collect()
torch.cuda.empty_cache()

print("GPU space cleared! Ready to train Afro-XLMR.")

GPU space cleared! Ready to train Afro-XLMR.


# Cell 19: Train the Augmented Model  
This will take roughly twice as long as the baseline because the dataset is now double the size (~8-10 minutes on a T4 GPU).

In [26]:
print("Starting training on Augmented Hausa Data (Condition B - 2x)...")
trainer_aug.train()

Starting training on Augmented Hausa Data (Condition B - 2x)...


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Accuracy
1,No log,0.331278,0.608562,0.629243,0.469101
2,0.367542,0.270110,0.708753,0.701595,0.469101
3,0.367542,0.268337,0.693672,0.690244,0.491573
4,0.226997,0.269678,0.720481,0.714286,0.522472
5,0.226997,0.324613,0.683642,0.685504,0.502809
6,0.134513,0.327565,0.698247,0.695444,0.508427
7,0.134513,0.339287,0.714289,0.711584,0.511236
8,0.069882,0.352865,0.710799,0.707581,0.508427
9,0.069882,0.358135,0.715811,0.711701,0.514045
10,0.041527,0.359013,0.712748,0.708585,0.514045


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=2690, training_loss=0.15874180474689015, metrics={'train_runtime': 1940.4528, 'train_samples_per_second': 22.108, 'train_steps_per_second': 1.386, 'total_flos': 2821967414323200.0, 'train_loss': 0.15874180474689015, 'epoch': 10.0})

# Cell 20: Final Evaluation and Comparison  
Now, we compare the augmented performance against your 0.6655 baseline.

In [27]:
print("\nEvaluating Augmented Model on Test Set...")
aug_results = trainer_aug.evaluate(augmented_bundle['test'])

print("\n" + "="*30)
print("EXPERIMENT RESULTS: HAUSA")
print(f"Baseline F1-Macro:  0.6655")
print(f"Condition B (2x) F1-Macro: {aug_results['eval_f1_macro']:.4f}")
print("="*30)

improvement = aug_results['eval_f1_macro'] - 0.6655
if improvement > 0:
    print(f"Success! Augmentation improved the model by {improvement:.4f}")
else:
    print("No improvement. The model might need more diverse pivot languages or paraphrasing.")


Evaluating Augmented Model on Test Set...



EXPERIMENT RESULTS: HAUSA
Baseline F1-Macro:  0.6655
Condition B (2x) F1-Macro: 0.6677
Success! Augmentation improved the model by 0.0022


# Cell 21: Re-Load Translation Model & Define Condition C Engine

In [28]:
# Cell 21: Re-Load NLLB & Define Condition C Translation Engine
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print("Re-loading NLLB Model for Condition C...")
model_name = "facebook/nllb-200-distilled-600M"
model_nllb = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.float16).to("cuda")
tokenizer_nllb = AutoTokenizer.from_pretrained(model_name)

def translate_nllb_v2(text, src_lang_code, tgt_lang_code):
    tokenizer_nllb.src_lang = src_lang_code
    inputs = tokenizer_nllb(text, return_tensors="pt", padding=True, truncation=True, max_length=400).to("cuda")
    tgt_lang_id = tokenizer_nllb.convert_tokens_to_ids(tgt_lang_code)

    # Using sampling for paraphrasing effect
    translated_tokens = model_nllb.generate(
        **inputs,
        forced_bos_token_id=tgt_lang_id,
        max_length=400,
        do_sample=True,
        top_k=50,
        top_p=0.95
    )
    return tokenizer_nllb.batch_decode(translated_tokens, skip_special_tokens=True)[0]

def get_condition_c(text):
    if not text or text.strip() == "":
        return text, text
    try:
        # 1. French Pivot (Hausa -> French -> Hausa)
        french_pivot = translate_nllb_v2(text, "hau_Latn", "fra_Latn")
        back_from_french = translate_nllb_v2(french_pivot, "fra_Latn", "hau_Latn")

        # 2. Hausa Paraphrase (Hausa -> English -> Hausa with sampling)
        english_pivot = translate_nllb_v2(text, "hau_Latn", "eng_Latn")
        paraphrase_hau = translate_nllb_v2(english_pivot, "eng_Latn", "hau_Latn")

        return back_from_french, paraphrase_hau
    except Exception as e:
        return text, text # Fallback if it fails
print("Condition C Engine Ready!")

Re-loading NLLB Model for Condition C...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Condition C Engine Ready!


# Cell 22: Generate the Data & Separate into Distinct Conditions

In [29]:
# Cell 22: Generate and Organize Condition C & Condition D Datasets
import copy
import pandas as pd
from tqdm.auto import tqdm

print("Starting Data Generation Pipeline...")

french_aug_data = []
para_aug_data = []

# Iterating over the ORIGINAL training data
for _, row in tqdm(df_hausa_train.iterrows(), total=len(df_hausa_train)):
    fr_text, para_text = get_condition_c(row['text'])

    # Create French back-translated row
    fr_row = copy.deepcopy(row)
    fr_row['text'] = fr_text
    french_aug_data.append(fr_row)

    # Create Paraphrased row
    para_row = copy.deepcopy(row)
    para_row['text'] = para_text
    para_aug_data.append(para_row)

df_french = pd.DataFrame(french_aug_data)
df_para = pd.DataFrame(para_aug_data)

# === STRUCTURAL ISOLATION OF CONDITIONS ===

# Condition C (2x): Original (1x) + Paraphrased (1x)
df_hausa_cond_c = pd.concat([df_hausa_train, df_para], ignore_index=True)

# Condition D (4x): Original (1x) + English BT (1x) + French BT (1x) + Paraphrased (1x)
df_hausa_cond_d = pd.concat([df_hausa_train, df_augmented, df_french, df_para], ignore_index=True)

print(f"Condition C Dataset Size (Paraphrasing Only): {len(df_hausa_cond_c)} rows")
print(f"Condition D Dataset Size (Combined BT + Paraphrasing): {len(df_hausa_cond_d)} rows")

# Save backups
df_hausa_cond_c.to_csv("hausa_augmented_cond_c_paraphrase.csv", index=False)
df_hausa_cond_d.to_csv("hausa_augmented_cond_d_combined.csv", index=False)

Starting Data Generation Pipeline...


  0%|          | 0/2145 [00:00<?, ?it/s]

Condition C Dataset Size (Paraphrasing Only): 4290 rows
Condition D Dataset Size (Combined BT + Paraphrasing): 8580 rows


# Cell 23: Clear GPU Memory

In [30]:
# Cell 23: Clear GPU Memory before training
import gc

if 'model_nllb' in globals():
    del model_nllb
if 'tokenizer_nllb' in globals():
    del tokenizer_nllb

gc.collect()
torch.cuda.empty_cache()

print("GPU space cleared! Ready to train Afro-XLMR pipelines.")

GPU space cleared! Ready to train Afro-XLMR pipelines.


# Cell 24: Re-tokenize and Re-bundle Separate Pipelines

In [31]:
# Cell 24: Re-bundle Datasets for Condition C and Condition D Separately
from datasets import Dataset, DatasetDict

# 1. Prepare Condition C (Paraphrasing Only)
dataset_c = Dataset.from_pandas(df_hausa_cond_c)
tokenized_c = dataset_c.map(tokenize_function, batched=True)
tokenized_c.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

bundle_cond_c = DatasetDict({
    'train': tokenized_c,
    'dev': tokenized_hausa['dev'],
    'test': tokenized_hausa['test']
})

# 2. Prepare Condition D (Combined)
dataset_d = Dataset.from_pandas(df_hausa_cond_d)
tokenized_d = dataset_d.map(tokenize_function, batched=True)
tokenized_d.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

bundle_cond_d = DatasetDict({
    'train': tokenized_d,
    'dev': tokenized_hausa['dev'],
    'test': tokenized_hausa['test']
})

print("Datasets mapped and ready!")

Map:   0%|          | 0/4290 [00:00<?, ? examples/s]

Map:   0%|          | 0/8580 [00:00<?, ? examples/s]

Datasets mapped and ready!


# Cell 25: Train and Evaluate Condition C (Paraphrasing)

In [32]:
# Cell 25: Train and Evaluate Condition C (Paraphrasing Only)
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

print("--- TRAINING CONDITION C (PARAPHRASING ONLY) ---")
model_cond_c = AutoModelForSequenceClassification.from_pretrained(
    "Davlan/afro-xlmr-base", num_labels=6, problem_type="multi_label_classification"
)

training_args_c = TrainingArguments(
    output_dir='./hausa_cond_c_results',
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    learning_rate=3e-5, lr_scheduler_type="cosine", per_device_train_batch_size=16,
    num_train_epochs=10, weight_decay=0.01, load_best_model_at_end=True, metric_for_best_model="f1_macro"
)

trainer_cond_c = Trainer(
    model=model_cond_c, args=training_args_c,
    train_dataset=bundle_cond_c['train'], eval_dataset=bundle_cond_c['dev'], compute_metrics=compute_metrics
)

trainer_cond_c.train()
cond_c_results = trainer_cond_c.evaluate(bundle_cond_c['test'])

--- TRAINING CONDITION C (PARAPHRASING ONLY) ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Accuracy
1,No log,0.322469,0.626944,0.640205,0.438202
2,0.382112,0.270830,0.694007,0.691329,0.474719
3,0.382112,0.258181,0.723905,0.721504,0.502809
4,0.258157,0.258672,0.705116,0.702509,0.502809
5,0.258157,0.247265,0.733029,0.733581,0.530899
6,0.173339,0.273636,0.710273,0.707986,0.497191
7,0.173339,0.283740,0.725265,0.721823,0.511236
8,0.111449,0.285432,0.733315,0.732934,0.525281
9,0.111449,0.287189,0.722544,0.721750,0.516854
10,0.082248,0.288103,0.717596,0.717762,0.514045


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

# Cell 26: Train and Evaluate Condition D (Combined)

In [33]:
# Cell 26: Train and Evaluate Condition D (Combined BT + Paraphrasing)
import gc

# Clean memory from previous run
del model_cond_c, trainer_cond_c
gc.collect()
torch.cuda.empty_cache()

print("\n--- TRAINING CONDITION D (COMBINED BT + PARAPHRASING) ---")
model_cond_d = AutoModelForSequenceClassification.from_pretrained(
    "Davlan/afro-xlmr-base", num_labels=6, problem_type="multi_label_classification"
)

training_args_d = TrainingArguments(
    output_dir='./hausa_cond_d_results',
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    learning_rate=3e-5, lr_scheduler_type="cosine", per_device_train_batch_size=16,
    num_train_epochs=10, weight_decay=0.01, load_best_model_at_end=True, metric_for_best_model="f1_macro"
)

trainer_cond_d = Trainer(
    model=model_cond_d, args=training_args_d,
    train_dataset=bundle_cond_d['train'], eval_dataset=bundle_cond_d['dev'], compute_metrics=compute_metrics
)

trainer_cond_d.train()
cond_d_results = trainer_cond_d.evaluate(bundle_cond_d['test'])


--- TRAINING CONDITION D (COMBINED BT + PARAPHRASING) ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Accuracy
1,0.393758,0.297804,0.661281,0.656073,0.404494
2,0.310206,0.276403,0.672363,0.671446,0.471910
3,0.255121,0.266760,0.699987,0.697076,0.502809
4,0.209256,0.288673,0.699643,0.699647,0.511236
5,0.159010,0.315759,0.699770,0.699415,0.500000
6,0.123751,0.323042,0.698236,0.697115,0.511236
7,0.092620,0.339721,0.707303,0.707466,0.522472
8,0.073427,0.356297,0.708752,0.704600,0.519663
9,0.057575,0.358727,0.714541,0.710234,0.533708
10,0.051960,0.361448,0.716532,0.712531,0.533708


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

# Cell 27: Final Structured Experiment Report

In [34]:
# Cell 27: Final Structured Experiment Report
print("\n" + "="*45)
print("       FINAL EXPERIMENTAL RESULTS: HAUSA")
print("="*45)
print(f"Condition A: Baseline (No Aug)     |  0.6655")
print(f"Condition B: Back-Translation Only  |  0.6677")
print(f"Condition C: Paraphrasing Only      |  {cond_c_results['eval_f1_macro']:.4f}")
print(f"Condition D: Combined (BT + Para)   |  {cond_d_results['eval_f1_macro']:.4f}")
print("="*45)

# Analytics summary logic
if cond_d_results['eval_f1_macro'] > max(0.6677, cond_c_results['eval_f1_macro']):
    print("Conclusion: Combining strategies (Condition D) produces the most robust performance gains.")
elif cond_c_results['eval_f1_macro'] > 0.6677:
    print("Conclusion: Targeted paraphrasing outperforms back-translation variations.")
else:
    print("Conclusion: Syntactic variance via back-translation remains the optimal individual approach.")


       FINAL EXPERIMENTAL RESULTS: HAUSA
Condition A: Baseline (No Aug)     |  0.6655
Condition B: Back-Translation Only  |  0.6677
Condition C: Paraphrasing Only      |  0.6559
Condition D: Combined (BT + Para)   |  0.6641
Conclusion: Syntactic variance via back-translation remains the optimal individual approach.
